# DTD 港交所交易日连续性测试

本 notebook 调用 `dtd_calendar_test_module.py`，按单日方式处理 temporary Input，并把成功结果写入独立的 temporary Output。

核心规则：

1. 只有 `China_HK_Trading_Calendar.xlsx` 中 `HK_Open=True` 的日期可产生 DTD。
2. 休市日返回 `SKIPPED_HK_CLOSED`，不会写 Output。
3. 开市日必须等于最后完成 DTD 日期之后的下一港交所交易日。
4. 同日重复、跳过开市日、日历覆盖不足或缺少目标日 Input 都会报错。
5. temporary Output 的列固定为 `Comp_no`、`Date`、`DTD`。

## 1 真实文件参数

默认假设：

- `vanke.xlsx` 保存 confirmed Input 和 Output。
- `data_temporary.xlsx` 保存与 Vanke Input 完全相同结构的增量 Input。
- `temporary_output.xlsx` 保存测试输出；第一次成功计算时自动创建。
- `TARGET_DATE` 每次只填一天，例如 `20251215`。

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_DIR = Path(r"C:\Users\susul\Desktop\苏苏重要文件备份\NUS CRI")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from dtd_calendar_test_module import (
    DTDTestError,
    MissingTradingDayError,
    process_daily_dtd,
    read_temporary_output,
)

CONFIRMED_FILE = PROJECT_DIR / "vanke.xlsx"
CALENDAR_FILE = PROJECT_DIR / "China_HK_Trading_Calendar.xlsx"
TEMPORARY_INPUT_FILE = PROJECT_DIR / "data_temporary.xlsx"
TEMPORARY_OUTPUT_FILE = PROJECT_DIR / "temporary_output.xlsx"

TARGET_DATE = "20251215"
COMPANY = None
RUN_REAL_FILE = False  # 确认文件路径和 TARGET_DATE 后改为 True

## 2 每天只运行一次

In [2]:
if RUN_REAL_FILE:
    daily_result = process_daily_dtd(
        TARGET_DATE,
        confirmed_file=CONFIRMED_FILE,
        temporary_input_file=TEMPORARY_INPUT_FILE,
        temporary_output_file=TEMPORARY_OUTPUT_FILE,
        calendar_file=CALENDAR_FILE,
        company=COMPANY,
    )
    display(pd.Series(daily_result.as_dict(), name="value"))
    display(read_temporary_output(TEMPORARY_OUTPUT_FILE))
else:
    print("RUN_REAL_FILE=False：本格没有修改任何真实 temporary 文件。")

RUN_REAL_FILE=False：本格没有修改任何真实 temporary 文件。


## 3 独立示例测试

下面的示例在系统临时目录内创建测试文件，不会修改项目中的真实文件。示例 Input 包含 20251213 至 20251217；市场价值和利率仅为测试假设。

运行顺序是：13、14、15、故意跳到17、再运行16、最后重新运行17。

In [3]:
from tempfile import TemporaryDirectory

import numpy as np


def make_demo_input(output_path: Path) -> None:
    confirmed = pd.read_excel(CONFIRMED_FILE, sheet_name="Input")
    last = confirmed.tail(1).copy()
    assumptions = {
        20251213: (61977.573739, 2.46),
        20251214: (61977.573739, 2.46),
        20251215: (62000.0, 2.45),
        20251216: (61500.0, 2.44),
        20251217: (61000.0, 2.43),
    }
    rows = []
    for date, (market_cap, risk_free_rate) in assumptions.items():
        row = last.copy()
        row["Date"] = date
        row["CUR_MKT_CAP(HKD)"] = market_cap
        row["Risk_Free_Rate"] = risk_free_rate
        rows.append(row)
    pd.concat(rows, ignore_index=True).to_excel(output_path, sheet_name="Input", index=False)


test_rows = []
with TemporaryDirectory() as test_directory:
    test_directory = Path(test_directory)
    demo_input = test_directory / "data_temporary.xlsx"
    demo_output = test_directory / "temporary_output.xlsx"
    make_demo_input(demo_input)

    for date in ("20251213", "20251214", "20251215"):
        result = process_daily_dtd(
            date,
            confirmed_file=CONFIRMED_FILE,
            temporary_input_file=demo_input,
            temporary_output_file=demo_output,
            calendar_file=CALENDAR_FILE,
        )
        test_rows.append(
            {
                "输入日期": date,
                "结果": result.status,
                "DTD": result.dtd,
                "是否写入": result.wrote_output,
                "说明": result.message,
            }
        )

    # 故意跳过 20251216，直接请求 20251217。
    try:
        process_daily_dtd(
            "20251217",
            confirmed_file=CONFIRMED_FILE,
            temporary_input_file=demo_input,
            temporary_output_file=demo_output,
            calendar_file=CALENDAR_FILE,
        )
    except MissingTradingDayError as error:
        test_rows.append(
            {
                "输入日期": "20251217",
                "结果": "ERROR_MISSING_TRADING_DAY",
                "DTD": np.nan,
                "是否写入": False,
                "说明": str(error),
            }
        )

    for date in ("20251216", "20251217"):
        result = process_daily_dtd(
            date,
            confirmed_file=CONFIRMED_FILE,
            temporary_input_file=demo_input,
            temporary_output_file=demo_output,
            calendar_file=CALENDAR_FILE,
        )
        test_rows.append(
            {
                "输入日期": date,
                "结果": result.status,
                "DTD": result.dtd,
                "是否写入": result.wrote_output,
                "说明": result.message,
            }
        )

    demo_temporary_output = read_temporary_output(demo_output)

test_results = pd.DataFrame(test_rows)
display(test_results)
print("最终 temporary Output：")
display(demo_temporary_output)

,输入日期,结果,DTD,是否写入,说明
0,20251213,SKIPPED_HK_CLOSED,NaN,False,2025-12-13 港交所休市 (Weekend)，不计算也不写入 DTD。
1,20251214,SKIPPED_HK_CLOSED,NaN,False,2025-12-14 港交所休市 (Weekend)，不计算也不写入 DTD。
2,20251215,DTD_WRITTEN,1.370494,True,2025-12-15 是下一港交所交易日；DTD=1.3704935021，已写入 temp...
3,20251217,ERROR_MISSING_TRADING_DAY,NaN,False,不能处理 2025-12-17；上一完成日是 2025-12-15，必须先处理下一港交所交易...
4,20251216,DTD_WRITTEN,1.356902,True,2025-12-16 是下一港交所交易日；DTD=1.3569017738，已写入 temp...
5,20251217,DTD_WRITTEN,1.342978,True,2025-12-17 是下一港交所交易日；DTD=1.3429779489，已写入 temp...


最终 temporary Output：


,Comp_no,Date,DTD
0,5338,20251215,1.370494
1,5338,20251216,1.356902
2,5338,20251217,1.342978


## 4 预期解释

- 20251213、20251214：周末，跳过，不写 Output。
- 20251215：12日之后的下一港交所交易日，计算并写入。
- 第一次请求20251217：报错，因为20251216尚未完成。
- 20251216：计算并写入。
- 第二次请求20251217：现在连续，计算并写入。

最终 temporary Output 应仅包含 20251215、20251216、20251217。